# 统计排序与函数应用

学习目标：为小表选择清楚的统计口径，完成稳定排序，并根据输入和输出要求选择聚合、逐元素处理或变换。

前置知识：Series 与 DataFrame、行列标签、轴、缺失值、Python 函数与条件表达式。

运行环境：Python 3.12、pandas 3.0；示例按 pandas 3.0.6 的接口行为编写。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

示例使用自制数据；后续单元沿用已导入的 pd。数值缺失示例显式使用 float64，以 NaN 表示缺失。

本环境已安装 PyArrow，默认 str 字符串采用 PyArrow 存储；频数示例中的显式 string 在本环境也采用该后端。

标明“预期异常”的单元会直接显示原始报错；阅读异常类型与原因后，继续运行下一单元。

## 1 汇总两次测量

先回答一个具体问题：两次测量各有多少有效记录，合计和平均值分别是多少？count 统计非缺失值，sum 求和，mean 求平均值。下面的数值单位都是摄氏度，缺失表示该次未取得读数。

DataFrame 统计默认 axis=0，沿行方向汇总，每列得到一个结果。先明确需要统计的列，可以避免把编号等无关数值混入结果。

In [1]:
import pandas as pd

measurements = pd.DataFrame(
    {"morning_c": [18.0, 20.0, None, 22.0],
     "afternoon_c": [20.0, 22.0, 24.0, None]},
    index=["R1", "R2", "R3", "R4"], dtype="float64",
)
summary = pd.DataFrame({
    "count": measurements.count(),
    "sum_c": measurements.sum(skipna=True),
    "mean_c": measurements.mean(skipna=True),
})
print(measurements)  # 四行两列，每列各有一个 NaN。
print(summary)  # 两列都有效 3 次；合计 60、66，平均值 20、22。
print(summary.dtypes)  # count 为 int64，sum_c、mean_c 为 float64。
print(summary.shape)  # 预期：(2, 3)，原测量列名成为汇总表行标签。

    morning_c  afternoon_c
R1       18.0         20.0
R2       20.0         22.0
R3        NaN         24.0
R4       22.0          NaN
             count  sum_c  mean_c
morning_c        3   60.0    20.0
afternoon_c      3   66.0    22.0
count       int64
sum_c     float64
mean_c    float64
dtype: object
(2, 3)


## 2 数量、轴与缺失口径

### 2.1 count 与 size

size 是属性，不加括号；Series.size 是元素数，DataFrame.size 是行数乘列数，包含缺失位置。它不等于 DataFrame 的行数。count() 则排除缺失值，并可指定按行或按列统计。

下面继续使用 measurements；axis=1 沿列方向汇总，每行得到一个结果。

In [2]:
print(measurements.size, len(measurements))  # 预期：8 4，单元格数与行数不同。
print(measurements["morning_c"].size, measurements["morning_c"].count())  # 4 3。
print(measurements.count(axis=1))  # R1、R2 为 2；R3、R4 为 1，dtype 为 int64。
print(measurements.mean(axis=1, skipna=True))
# R1 至 R4 分别为 19、21、24、22，dtype 为 float64。
# 后两行只基于一次有效测量，不应当成“两次测量的完整平均值”。

8 4
4 3
R1    2
R2    2
R3    1
R4    1
dtype: int64
R1    19.0
R2    21.0
R3    24.0
R4    22.0
dtype: float64


### 2.2 skipna 与全部缺失

sum、mean 默认 skipna=True，计算时跳过缺失值；skipna=False 会让含缺失的汇总结果也缺失。跳过缺失不是把缺失填成 0。

sum 默认 min_count=0，所以空序列或全部缺失的序列仍可得到 0。若任务要求至少一个有效值，设置 min_count=1。下面先沿用 measurements，再单独检查全部缺失的输入。

In [3]:
print(measurements.mean(skipna=False))  # 两列均为 NaN。
missing = pd.Series([None, None], dtype="float64")
print(missing.sum(), missing.sum(min_count=1), missing.mean())  # 预期：0.0 nan nan。
print(missing.count())  # 预期：0；不能把默认合计 0 解释成真的测到了零。

morning_c     NaN
afternoon_c   NaN
dtype: float64
0.0 nan nan
0


### 2.3 numeric_only 的范围

DataFrame.mean 的 numeric_only 默认 False，不会自动略过所有不适合求均值的列。设为 True 时，浮点、整数和布尔列都可参与；它按类型选择，不理解哪些列在业务上适合统计。

下面用独立的混合表说明：enabled 的均值是 True 所占比例，不能和温度看成相同指标。

In [4]:
mixed = pd.DataFrame({"sensor": ["A", "B"], "temperature_c": [18.0, 22.0],
                      "enabled": [True, False]})
print(mixed.mean(numeric_only=True))  # temperature_c 为 20.0，enabled 为 0.5。
print(mixed[["temperature_c"]].mean())  # 只计算明确选出的温度列。

# 预期 TypeError：numeric_only=False 包含 sensor 字符串列，无法对它求均值。
mixed.mean(numeric_only=False)

temperature_c    20.0
enabled           0.5
dtype: float64
temperature_c    20.0
dtype: float64


TypeError: Cannot perform reduction 'mean' with string dtype

## 3 标准差与分位数

### 3.1 std 的分母

标准差描述数值相对平均值的离散程度。std 默认 ddof=1，计算方差时使用 N−1 作分母；ddof=0 使用 N。这里 N 是参与计算的有效值个数，ddof 是分母中减去的自由度调整量。

下面三个有效值的均值是 20，离均差平方和为 8。采用哪个分母取决于统计口径；报告时应注明。有效数量不足以形成正分母时，不能得到有效标准差。

In [5]:
values = pd.Series([18.0, 20.0, 22.0, None], dtype="float64")
print(values.std(ddof=1))  # 预期：2.0，即 sqrt(8 / 2)。
print(values.std(ddof=0))  # 约 1.633，即 sqrt(8 / 3)。
one_value = pd.Series([20.0], dtype="float64")
print(one_value.std(ddof=1), one_value.std(ddof=0))  # 预期：nan 0.0。
print(values.std(skipna=False))  # 预期：nan，缺失不再被跳过。

2.0
1.632993161855452
nan 0.0
nan


### 3.2 quantile 与 describe

quantile 的 q 取 0 至 1，表示分位位置，例如 0.5 表示中位数；默认在相邻有序值之间做线性插值。传一个 q 得到 Series，传 q 列表得到以这些分位位置为索引的 DataFrame。

describe 提供快速摘要：数值列包括有效数量、均值、标准差、最小值、分位数与最大值，排除缺失。混合表默认摘要数值列，可用 include="all" 纳入其他类型。下面单独使用四个等间距成绩，方便手算插值。

In [6]:
scores = pd.DataFrame({"score": [60.0, 70.0, 80.0, 90.0]}, index=list("ABCD"))
print(scores.quantile(0.5))  # score 为 75.0，结果是一维 Series。
print(scores.quantile([0.25, 0.5, 0.75]))  # 依次 67.5、75.0、82.5，形状 (3, 1)。
print(scores.quantile(0.25, interpolation="lower"))  # 预期：60.0，改用较低相邻值。
print(scores.describe())  # count 为 4、mean 为 75；std 使用 ddof=1。
print(scores.describe().dtypes)  # score 摘要列为 float64，count 显示为 4.0。

score    75.0
Name: 0.5, dtype: float64
      score
0.25   67.5
0.50   75.0
0.75   82.5
score    60.0
Name: 0.25, dtype: float64
           score
count   4.000000
mean   75.000000
std    12.909944
min    60.000000
25%    67.500000
50%    75.000000
75%    82.500000
max    90.000000
score    float64
dtype: object


## 4 频数与不同值数量

value_counts 返回每个值出现的次数，默认按频数降序且排除缺失；normalize=True 改为相对频率。nunique 只返回不同值的数量。两者用 dropna=False 把缺失也纳入口径。

pandas 3.0 的 value_counts 频数排序是稳定的，同频值保持原数据中的先后次序。下面显式用 string 类型和 pd.NA，展示缺失是否计入分母。

In [7]:
groups = pd.Series(["B", "A", "B", "A", pd.NA], dtype="string", name="group")
print(groups.value_counts())  # B、A 各 2 次；同频时 B 在前。
print(groups.value_counts(normalize=True))  # 排除缺失，以 4 为分母，各 0.5。
print(groups.value_counts(normalize=True, dropna=False))  # B、A 各 0.4，<NA> 为 0.2。
print(groups.nunique(), groups.nunique(dropna=False))  # 预期：2 3。
print(groups.describe())  # count 为 4、unique 为 2、freq 为 2；并列 top 不保证取哪一个。
# 本环境频数 dtype 为 int64[pyarrow]，比例为 double[pyarrow]；数值口径不因此改变。

group
B    2
A    2
Name: count, dtype: int64[pyarrow]
group
B    0.5
A    0.5
Name: proportion, dtype: double[pyarrow]
group
B       0.4
A       0.4
<NA>    0.2
Name: proportion, dtype: double[pyarrow]
2 3
count     4
unique    2
top       B
freq      2
Name: group, dtype: object


## 5 按值排序与按标签排序

### 5.1 同分保持输入顺序

sort_values 按列值排列行，ascending=False 为降序，na_position 决定缺失值放首尾。单列排序中指定 kind="stable"，可以保证相同值的记录保持输入顺序；默认 quicksort 不作这个保证。

下面的输入顺序代表登记先后，要求同分时保留先登记者在前。排序默认返回新表，行标签随记录移动。

In [8]:
ranking = pd.DataFrame(
    {"score": [80.0, 90.0, 90.0, None], "name": ["甲", "乙", "丙", "丁"]},
    index=["R3", "R2", "R1", "R4"],
)
ordered = ranking.sort_values("score", ascending=False, kind="stable", na_position="last")
print(ordered)  # R2、R1 同为 90，保持原先次序；随后 R3，缺失 R4 在末尾。
print(ordered.index.tolist(), ordered.shape)  # ['R2', 'R1', 'R3', 'R4'] (4, 2)。
print(ranking.index.tolist())  # 原表仍为 R3、R2、R1、R4。
assert ordered.index.tolist() == ["R2", "R1", "R3", "R4"]

    score name
R2   90.0    乙
R1   90.0    丙
R3   80.0    甲
R4    NaN    丁
['R2', 'R1', 'R3', 'R4'] (4, 2)
['R3', 'R2', 'R1', 'R4']


### 5.2 多列条件与 sort_index

若同分时改按其他字段排序，把多个列名交给 sort_values，并给出各自升降序。kind 对 DataFrame 的单列或单标签排序生效，不应把它当成多列排序的控制开关。

sort_index 则按轴标签排序，默认对行标签操作。下面继续使用 ranking；按标签排序不等于恢复原始登记顺序。

In [9]:
by_two_keys = ranking.sort_values(["score", "name"], ascending=[False, True])
print(by_two_keys.index.tolist())  # 本例同分的“丙”排在“乙”前：R1、R2、R3、R4。
print(ranking.sort_index())  # 行标签按 R1、R2、R3、R4 排列，记录内容跟随标签。
print(ranking.sort_index(axis=1).columns.tolist())  # 预期：['name', 'score']。
print(by_two_keys.dtypes)  # score 为 float64，name 为 str；排序没有改变列类型。

['R1', 'R2', 'R3', 'R4']
    score name
R1   90.0    丙
R2   90.0    乙
R3   80.0    甲
R4    NaN    丁
['name', 'score']
score    float64
name         str
dtype: object


## 6 逐元素处理与按轴应用

### 6.1 Series.map 与字典映射

Series.map 可用字典把每个值换成对应值，保留原索引；普通字典没有包含的键会得到缺失。也可以传一个“单值输入、单值输出”的函数。

下面把状态码变成中文名称，先检查未被映射的状态，不把缺失误认为合法名称。

In [10]:
status = pd.Series(["ok", "hold", "other"], index=["A", "B", "C"])
labels = status.map({"ok": "通过", "hold": "待定"})
print(labels)  # A 为通过，B 为待定，C 为 NaN；pandas 3 默认字符串结果为 str。
print(labels.isna().tolist())  # 预期：[False, False, True]。
print(labels.index.equals(status.index))  # 预期：True。

A     通过
B     待定
C    NaN
dtype: str
[False, False, True]
True


### 6.2 DataFrame.map 与内置运算

DataFrame.map 把函数应用到每个元素，函数接收和返回单个值；na_action="ignore" 可保留缺失而不把它传给函数。当前接口使用 map，旧名称 applymap 已在 pandas 3 移除。

如果已有等价的列运算或内置方法，优先使用它们。下面对比两种“每个值加 1”的写法，结果相同，不需要自定义函数才能完成。

In [11]:
small = pd.DataFrame({"a": [1, 2], "b": [3, 4]}, index=["R1", "R2"])
mapped = small.map(lambda value: value + 1)
direct = small + 1
print(mapped)  # a 为 2、3；b 为 4、5；标签与 (2, 2) 形状不变。
print(mapped.equals(direct))  # 预期：True。
print(mapped.dtypes)  # 两列仍为 int64。

    a  b
R1  2  4
R2  3  5
True
a    int64
b    int64
dtype: object


### 6.3 apply 的输入是一行或一列

DataFrame.apply 在默认 raw=False 时，把每列或每行作为 Series 交给函数：axis=0 逐列，axis=1 逐行。函数返回标量时通常汇成一个 Series；返回结构改变时，最终结果结构也会改变，不能把 apply 一律理解成逐元素调用。

下面继续使用 small，函数计算传入 Series 的最大值与最小值之差。不要在函数内修改传入对象；这里用 apply 演示接口，相同任务也可由内置 max、min 相减完成。

In [12]:
def value_range(values):
    return values.max() - values.min()


column_ranges = small.apply(value_range, axis=0)
row_ranges = small.apply(value_range, axis=1)
print(column_ranges)  # a、b 的极差均为 1；索引来自原列名。
print(row_ranges)  # R1、R2 的极差均为 2；索引来自原行标签。
print(row_ranges.equals(small.max(axis=1) - small.min(axis=1)))  # 预期：True。

a    1
b    1
dtype: int64
R1    2
R2    2
dtype: int64
True


## 7 聚合与保留长度的变换

### 7.1 agg 汇总多个指标

agg 是 aggregate 的简写，可接收函数名、函数列表或“列名 → 函数”的字典。对 DataFrame 的列做单个聚合，结果通常是 Series；多个聚合会形成 DataFrame。对 Series 做单个聚合则得到标量。

下面继续使用 small，同时生成 sum 和 mean，函数名明确使用 pandas 的统计操作。pandas 3 不再把传入的 NumPy 函数或 Python 内置函数自动替换为同名 pandas 实现；需要 pandas 口径时使用这里的字符串名称。

In [13]:
print(small.agg("sum"))  # 一维 Series：a 为 3、b 为 7，dtype 为 int64。
aggregated = small.agg(["sum", "mean"])
print(aggregated)  # 行为 sum、mean；a 为 3、1.5，b 为 7、3.5。
print(aggregated.shape, aggregated.dtypes.tolist())  # (2, 2)，两列为 float64。
print(small["a"].agg("sum"))  # 预期：3，标量。
print(small.agg({"a": "sum", "b": "max"}))  # 按列选指标：a 为 3、b 为 4。

a    3
b    7
dtype: int64


        a    b
sum   3.0  7.0
mean  1.5  3.5
(2, 2) [dtype('float64'), dtype('float64')]
3
a    3
b    4
dtype: int64


### 7.2 transform 保留观测位置

transform 用于变换而非压缩观测，要求返回结果保留被变换轴的长度；多个函数可以增加结果列，因此不应笼统说成“任何调用都与原表形状相同”。

下面用单个函数对 small 的每列减去本列均值，结果仍有每条原记录。相反，sum 把整列压缩成一个值，不符合这里的 transform 要求。

In [14]:
centered = small.transform(lambda column: column - column.mean())
print(centered)  # 每列 R1 为 -0.5、R2 为 0.5；两列为 float64。
print(centered.shape, centered.index.equals(small.index))  # (2, 2) True。
print(centered.columns.equals(small.columns))  # 预期：True。

# 预期 ValueError：sum 将每列缩减为一个值，不满足 transform 保留原形状的要求。
small.transform("sum")

      a    b
R1 -0.5 -0.5
R2  0.5  0.5
(2, 2) True
True


ValueError: Function did not transform

## 8 选学：排名、前几名与累计值
### 8.1 rank 与 nlargest

rank 给每条记录标名次，并保留原顺序；默认并列取平均名次，method="min" 取并列组的最小名次。ascending=False 表示值越大名次越靠前，缺失默认仍为缺失。

nlargest 取数值列最大的前 n 行；keep="all" 保留截止值处所有并列，因此返回行数可能大于 n。下面继续使用 ranking。

In [15]:
print(ranking["score"].rank(ascending=False))  # R3 为 3，R2、R1 为 1.5，R4 为 NaN。
print(ranking["score"].rank(ascending=False, method="min"))  # 并列第一变成 1.0。
print(ranking.nlargest(1, "score", keep="first").index.tolist())  # 预期：['R2']。
print(ranking.nlargest(1, "score", keep="all").index.tolist())  # 预期：['R2', 'R1']。

R3    3.0
R2    1.5
R1    1.5
R4    NaN
Name: score, dtype: float64
R3    3.0
R2    1.0
R1    1.0
R4    NaN
Name: score, dtype: float64
['R2']
['R2', 'R1']


### 8.2 cumsum 与缺失

cumsum 按当前顺序累计，返回同样长度的 Series，不会先按标签自动排序。默认 skipna=True 时，缺失位置保持缺失，后续有效位置继续累计；False 时，缺失会传播到后续结果。

下面的顺序就是三天的记录次序，值表示每天新增件数。累计结果必须保留这项顺序约定。

In [16]:
daily = pd.Series([2.0, None, 3.0], index=["day1", "day2", "day3"], dtype="float64")
print(daily.cumsum(skipna=True))  # 2.0、NaN、5.0。
print(daily.cumsum(skipna=False))  # 2.0、NaN、NaN。
print(daily.cumsum().index.equals(daily.index))  # 预期：True。
# 第二天的缺失仍是未知；累计到 5.0 只汇总了现有有效记录。

day1    2.0
day2    NaN
day3    5.0
dtype: float64
day1    2.0
day2    NaN
day3    NaN
dtype: float64
True


## 9 选学：相关与协方差
### 9.1 有效配对与最少数量

corr 默认计算 Pearson 相关系数；cov 计算协方差。对同一组有效配对，Pearson 系数可理解为用两列标准差归一化的协方差，数值尺度不同于协方差。

它们按两列同时非缺失的记录配对，各列单独的 count 不能代替配对数。min_periods 指定每一对列至少需要多少条共同有效记录；不够时返回 NaN。下面只讨论两列的共同记录，不把其他列缺失也一并删除。

In [17]:
paired = pd.DataFrame(
    {"x": [1.0, 2.0, None, 4.0], "y": [2.0, None, 6.0, 8.0]},
    index=["A", "B", "C", "D"], dtype="float64",
)
complete = paired.dropna(subset=["x", "y"])
print(paired.count().tolist(), len(complete))  # 各有 3 个有效值，但共同配对仅 2 个。
print(complete.index.tolist())  # 预期：['A', 'D']，配对为 (1, 2)、(4, 8)。
print(paired.corr(min_periods=2).loc["x", "y"])  # 预期：1.0，仅基于这两个配对。
print(paired.cov(min_periods=2).loc["x", "y"])  # 预期：9.0，默认样本协方差。
print(paired.corr(min_periods=3).loc["x", "y"])  # 预期：nan。
print(paired.cov(min_periods=3).loc["x", "y"])  # 预期：nan。
# 不能把“表有四行”当作这两个结果使用了四个有效配对。

[3, 3] 2
['A', 'D']
1.0
9.0
nan
nan


### 9.2 协方差分母与标签对齐

DataFrame.cov 的 ddof 默认 1；文档限定这个参数只在表内没有 NaN 时适用。若需要明确比较其他分母，应先确定有效配对，再对无缺失的输入计算。下面沿用 complete，其两个配对的离均差乘积之和为 9。

Series.corr 还会按索引标签对齐，不能只看两个列表的排列顺序。常数序列的 Pearson 相关系数没有定义；缺失结果不应当作相关系数 0。

In [18]:
print(complete.cov(ddof=1).loc["x", "y"])  # 9 / (2 - 1) = 9.0。
print(complete.cov(ddof=0).loc["x", "y"])  # 9 / 2 = 4.5。
left = pd.Series([1.0, 2.0, 3.0], index=["A", "B", "C"])
right = pd.Series([1.0, 2.0, 3.0], index=["C", "B", "A"])
print(left.corr(right))  # 预期：-1.0；按 A、B、C 对齐后数值方向相反。

9.0
4.5
-1.0


## 本章小结

（1）统计先说明有效数量、缺失口径、轴和列范围；size 与 count 回答不同问题。

（2）标准差注明 ddof，分位数注明插值口径；numeric_only 不替代业务列选择。

（3）排序检查标签和并列顺序；单列需保持同值先后时显式使用稳定排序。

（4）map 逐元素处理，apply 按行列传入函数，agg 汇总指标，transform 保留观测长度。简单任务优先使用已有内置操作。

（5）选学的排名、累计、相关与协方差各有顺序、并列或有效配对条件，应结合输入解释结果。

## 练习

（1）先预测下面每个统计结果，再运行核对。分别解释 size、count、skipna 和 ddof 的作用。

In [19]:
exercise = pd.Series([2.0, None, 4.0], dtype="float64")
print(exercise.size, exercise.count())
print(exercise.mean(), exercise.mean(skipna=False))
print(exercise.std(ddof=0), exercise.std(ddof=1))
# 运行前记录预测；运行后用两个有效值及它们的平均值手算核对。

3 2
3.0 nan
1.0 1.4142135623730951


（2）为下面的两次数值记录建立汇总表，列出有效数量、均值与中位数。随后增加约束：只接受两次记录都有效的行均值，不允许仅凭一个值生成行均值。写出新的表达式并解释参数选择。

In [20]:
readings = pd.DataFrame({"first": [10.0, 20.0, None], "second": [14.0, None, 30.0]},
                        index=["A", "B", "C"], dtype="float64")
# 在此生成按列汇总，并单独生成满足新约束的按行均值。
# 检查：两列有效数量都为 2；均值与中位数分别为 15、22。
# 新约束下只有 A 行得到 12.0，B、C 为 NaN；保留原行标签。

（3）按成绩降序排列下表，同分保持登记顺序，缺失放在末尾。再改成按行标签排序，解释为何这次不再保留登记顺序。

In [21]:
entries = pd.DataFrame({"score": [80.0, 90.0, 90.0, None]}, index=["C", "B", "A", "D"])
# 在此显式选择排序参数，用输出标签顺序核对并列处理。
# 检查：按成绩为 B、A、C、D；按标签为 A、B、C、D。
# 原表不变，score 仍为 float64；解释稳定性要求对应哪个参数。

（4）下面三个任务分别适合内置算术、map、agg 还是 transform？完成它们并解释理由：把状态码翻译成中文；每列生成一个最大值；每个数值增加 2。

最后把第二项改成“保留每条观测，减去本列最小值”，说明输出要求为什么改变了接口选择。

In [22]:
codes = pd.Series(["ok", "hold"], index=["A", "B"])
numbers = pd.DataFrame({"x": [1, 3], "y": [4, 8]}, index=["A", "B"])
# 在此选择适合的写法，并用注释解释每项的输入与输出。
# 检查：翻译结果为通过、待定；按列最大值为 3、8；加 2 后 x 为 3、5，y 为 6、10。
# 改变第二项后应保留 (2, 2) 形状：x 为 0、2，y 为 0、4。
# 若用函数，函数只返回结果，不修改传入对象。

### 重点练习提示（第 2 题）

提示一：先区分按列统计与逐行求均值；再判断一个缺失是否允许被忽略。

提示二：按列用 agg 生成 count、mean、median；新约束的行均值需要 skipna=False。

### 参考解析（第 2 题）

readings.agg(["count", "mean", "median"]) 按列得到 first 的有效数 2、均值和中位数 15；second 对应为 2、22、22。readings.mean(axis=1, skipna=False) 得到 A=12.0，B、C 均缺失，索引仍为 A、B、C。默认跳过缺失会给 B、C 分别产生 20.0、30.0，但这不符合“两次记录都有效”的新约束。这里数值列固定只有两列，核对结果时还要确认没有误把按列均值当作按行均值。

## 参考与引用来源

在线文档可能随发布更新；固定版本对照见下表 pandas v3.0.6 文档源码。API 参数与异常仍须结合所列页面的具体定位阅读。

| 网站 | 本章参考内容与定位 |
| --- | --- |
| pandas 官方在线文档（课程基线 3.0.6） | [DataFrame.count](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.count.html)、[DataFrame.size](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.size.html)、[DataFrame.sum](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sum.html)、[DataFrame.mean](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html)、[DataFrame.std](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.std.html) 的 axis、skipna、numeric_only、min_count、ddof 与返回值；[DataFrame.quantile](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.quantile.html)、[DataFrame.describe](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.describe.html)、[Series.value_counts](https://pandas.pydata.org/docs/reference/api/pandas.Series.value_counts.html)、[Series.nunique](https://pandas.pydata.org/docs/reference/api/pandas.Series.nunique.html) 的插值、摘要、缺失口径及 value_counts 在 3.0 改为稳定排序；[DataFrame.sort_values](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_values.html)、[DataFrame.sort_index](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.sort_index.html) 的 kind、ascending、na_position 及单列稳定性条件；[Series.map](https://pandas.pydata.org/docs/reference/api/pandas.Series.map.html)、[DataFrame.map](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.map.html)、[DataFrame.apply](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.apply.html)、[DataFrame.agg](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.agg.html)、[DataFrame.transform](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.transform.html) 的输入、返回类型与禁止修改传入对象的 Notes；[pandas 3.0.0 release notes](https://pandas.pydata.org/docs/whatsnew/v3.0.0.html) 的 Enforced deprecations：移除 DataFrame.applymap，以及 apply、agg、transform 不再自动替换 NumPy 或内置函数；[DataFrame.rank](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rank.html)、[DataFrame.nlargest](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html)、[Series.cumsum](https://pandas.pydata.org/docs/reference/api/pandas.Series.cumsum.html) 的并列、截止值与 skipna；[DataFrame.corr](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.corr.html)、[DataFrame.cov](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.cov.html)、[Series.corr](https://pandas.pydata.org/docs/reference/api/pandas.Series.corr.html) 的有效配对、min_periods、ddof 适用范围、自动标签对齐与常数输入；[字符串迁移指南](https://pandas.pydata.org/docs/user_guide/migration-3-strings.html) 的 Background、For existing users of the nullable StringDtype：默认 str 后端与显式 string 的缺失语义。 |
| NumPy 官方文档（NumPy 2.5） | [corrcoef](https://numpy.org/doc/2.5/reference/generated/numpy.corrcoef.html) 的相关系数与协方差矩阵关系式，用于说明选学部分的归一化含义。 |
| GitHub 官方项目（版本化来源） | pandas v3.0.6 文档源码：[v3.0.0](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/whatsnew/v3.0.0.rst)、[migration-3-strings](https://github.com/pandas-dev/pandas/blob/v3.0.6/doc/source/user_guide/migration-3-strings.rst)；对应上列同名指南或发布说明的小节，作为固定版本对照。 |